## **Prompt caching**

Many providers offer prompt caching features to reduce latency and cost on repeat processing of the same tokens. Prompt caching is often only engaged above a minimum input token threshold. These features can be **implicit** or **explicit**: 
- **Implicit prompt caching:** providers will automatically pass on cost savings if a request hits a cache. Examples: OpenAI and Gemini.
- **Explicit caching:** providers allow you to manually indicate cache points for greater control or to guarantee cost savings. Example: ChatOpenAI (via prompt_cache_key)

Cache usage will be reflected in the usage metadata of the model response.
```python
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

response = model.invoke("Hello!")
response.usage_metadata
```
```output
{
 'input_tokens': 8,
 'output_tokens': 304,
 'total_tokens': 312,
 'input_token_details': {'audio': 0, 'cache_creation': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 256}
}
```


A cache is useful for two reasons:

1. It can save you money by reducing the number of API calls you make to the LLM provider if you're often requesting the same completion multiple times.
2. It can speed up your application by reducing the number of API calls you make to the LLM provider.


The cache interface consists of the following methods:

- lookup: Look up a value based on a `prompt` and `llm_string`.
- update: Update the cache based on a `prompt` and `llm_string`.
- clear: Clear the cache.

In addition, the cache interface provides an async version of each method.

The default implementation of the async methods is to run the synchronous
method in an executor. It's recommended to override the async methods
and provide async implementations to avoid unnecessary overhead.


llm_string: A string representation of the LLM configuration.
- This is used to capture the invocation parameters of the LLM (e.g., model name, temperature, stop tokens, max tokens, etc.).
- These invocation parameters are serialized into a string representation.

In [5]:
# !pip install langchain

In [11]:
from langchain_core.caches import InMemoryCache

# Initialize cache
cache = InMemoryCache(maxsize=100)

maxsize: The maximum number of items to store in the cache.
- If `None`, the cache has no maximum size.
- If the cache exceeds the maximum size, the oldest items are removed.

In [12]:
# from langchain_core.globals import set_llm_cache

# set_llm_cache(cache)

## **Calling LLM**

In [ ]:
%%time

prompt = "explain about the transformer algorithm"

response = llm.invoke(prompt)

print(response)

In [ ]:
%%time

prompt = "explain about the transformer algorithm"

response = llm.invoke(prompt)

print(response)

## **Rate Limiters**

In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # 1 request every 10s
    check_every_n_seconds=0.1,  # Check every 100ms whether allowed to make a request
    max_bucket_size=10,  # Controls the maximum burst size.
)

model = init_chat_model(
    model="gpt-5",
    model_provider="openai",
    rate_limiter=rate_limiter  
)

The provided rate limiter can only limit the number of requests per unit time. It will not help if you need to also limit based on the size of the requests.

## **Token Usage**

A number of model providers return token usage information as part of the invocation response. When available, this information will be included on the AIMessage objects produced by the corresponding model. For more details, see the messages guide.

You can track aggregate token counts across models in an application using either a callback or context manager, as shown below:

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import UsageMetadataCallbackHandler

model_1 = init_chat_model(model="gpt-4.1-mini")
model_2 = init_chat_model(model="claude-haiku-4-5-20251001")

callback = UsageMetadataCallbackHandler()
result_1 = model_1.invoke("Hello", config={"callbacks": [callback]})
result_2 = model_2.invoke("Hello", config={"callbacks": [callback]})
print(callback.usage_metadata)